# Yb171 `total_decay` get_nearby 分窗修复测试

这个 notebook 不修改 `rydcalc` 源码，而是在 notebook 里复刻 `total_decay` 的核心逻辑：

1. 用 `atom.get_nearby(...)` 动态生成候选终态；
2. 对终态调用 `atom.partial_decay(initial_state, final_state, env)`；
3. 累加 partial decay 得到总衰减率和寿命。

和原生 `total_decay` 的区别是：不把 `st.n` 直接作为一个巨大的 MQDT `nu` 半宽一次性传给 `get_nearby`，而是把物理上需要的 `nu` 区间切成小窗口。这样可以避开 slides 第 8-22 页中提到的 `[0.001, 129.121]` 巨大搜索区间和低 `nu` 数值病态问题。


In [1]:
from __future__ import annotations

import copy
import math
import os
import sys
import time
from pathlib import Path

import numpy as np

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-rydcalc")

from neutral_yb.external.rydcalc_adapter import build_yb171_atom, load_rydcalc  # noqa: E402

rydcalc = load_rydcalc(require_c_extension=True)
yb = build_yb171_atom(cpp_numerov=True, use_db=False, require_c_extension=True)
initial_state = yb.get_state((65, 0, 1.5, -1.5), tt="vlfm")

native_lo = initial_state.nub - initial_state.n
native_hi = initial_state.nub + initial_state.n
print("Target state")
print(f"  {initial_state}")
print(f"  st.n   = {initial_state.n:.8f}")
print(f"  st.nub = {initial_state.nub:.8f}")
print("Native total_decay would request one symmetric get_nearby interval")
print(f"  [{native_lo:.8f}, {native_hi:.8f}], width={native_hi - native_lo:.8f}")


Target state
  |171Yb:64.56,L=0,F=1.5,-1.5>
  st.n   = 64.56000000
  st.nub = 64.56106363
Native total_decay would request one symmetric get_nearby interval
  [0.00106363, 129.12106363], width=129.12000000


## 修复思路

`Ytterbium171.get_nearby` 对 MQDT 态的关键输入是 `st.nub` 和 `include_opts["dn"]`，实际搜索区间是

\[
[\mathrm{st.nub}-dn,\ \mathrm{st.nub}+dn].
\]

因此可以在 notebook 中构造一个浅拷贝 `proxy_state`，只临时改变 `proxy_state.nub` 为窗口中心，再把 `dn` 设成窗口半宽。这样仍然由 `get_nearby` 动态生成终态，但不会触发一次性的超大区间搜索。

这里使用 `dl=1, df=1, dm=1`，和原生 `total_decay` 对当前 S 态的 E1 终态扫描一致。`dipole_allowed=True` 只是提前去掉 partial rate 为零的非 E1 终态，不改变非零 decay rate 的求和结果。


In [2]:
def _finite_float(value) -> bool:
    try:
        return math.isfinite(float(value))
    except Exception:
        return False


def state_signature(state, *, energy_bin_hz: float = 1e6):
    """Key used to remove overlap from adjacent get_nearby windows."""
    return (
        round(state.get_energy_Hz() / energy_bin_hz),
        round(2 * state.f),
        round(2 * state.m),
        int(state.parity),
    )


def get_nearby_in_nu_window(atom, reference_state, *, nu_lo: float, nu_hi: float, dl=1, df=1, dm=1):
    """Call atom.get_nearby on a controlled MQDT effective-quantum-number window."""
    center = 0.5 * (nu_lo + nu_hi)
    half_width = 0.5 * (nu_hi - nu_lo)

    proxy_state = copy.copy(reference_state)
    proxy_state.nub = center
    proxy_state.v_exact = center
    proxy_state.n = center
    proxy_state.v = center

    return atom.get_nearby(
        proxy_state,
        include_opts={"dn": half_width, "dl": dl, "df": df, "dm": dm},
    )


def build_windowed_get_nearby_basis(
    atom,
    initial,
    *,
    nu_min: float = 3.0,
    nu_max: float | None = None,
    window_width: float = 5.0,
    window_overlap: float = 0.10,
    dl: int = 1,
    df: int = 1,
    dm: int = 1,
    dipole_allowed: bool = True,
):
    """Generate final states with the same get_nearby path as total_decay, but in small windows."""
    if nu_max is None:
        nu_max = initial.nub + 10.0

    unique = {}
    windows = []
    errors = []
    skipped = {"nonfinite_energy": 0, "not_dipole_allowed": 0, "duplicates": 0}

    start = time.perf_counter()
    lo = float(nu_min)
    while lo < nu_max - 1e-12:
        hi = min(float(nu_max), lo + float(window_width))
        query_lo = lo - 0.5 * window_overlap
        query_hi = hi + 0.5 * window_overlap
        window_start = time.perf_counter()
        try:
            states = get_nearby_in_nu_window(
                atom,
                initial,
                nu_lo=query_lo,
                nu_hi=query_hi,
                dl=dl,
                df=df,
                dm=dm,
            )
        except Exception as exc:
            states = []
            errors.append(
                {
                    "nu_lo": query_lo,
                    "nu_hi": query_hi,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )

        added = 0
        duplicate = 0
        for final_state in states:
            if final_state is None:
                continue
            if not _finite_float(final_state.get_energy_Hz()):
                skipped["nonfinite_energy"] += 1
                continue
            if dipole_allowed and not initial.allowed_multipole(final_state, k=1):
                skipped["not_dipole_allowed"] += 1
                continue
            key = state_signature(final_state)
            if key in unique:
                duplicate += 1
                skipped["duplicates"] += 1
                continue
            unique[key] = final_state
            added += 1

        windows.append(
            {
                "nu_lo": lo,
                "nu_hi": hi,
                "query_lo": query_lo,
                "query_hi": query_hi,
                "raw_states": len(states),
                "added_unique": added,
                "duplicates": duplicate,
                "seconds": time.perf_counter() - window_start,
            }
        )
        lo = hi

    return {
        "states": list(unique.values()),
        "windows": windows,
        "errors": errors,
        "skipped": skipped,
        "seconds": time.perf_counter() - start,
        "nu_min": nu_min,
        "nu_max": nu_max,
        "window_width": window_width,
        "window_overlap": window_overlap,
    }


def decay_from_windowed_basis(atom, initial, env, basis):
    """Compute total decay from a precomputed get_nearby basis."""
    rows = []
    partial_failures = []
    start = time.perf_counter()
    for final_state in basis["states"]:
        try:
            gamma = atom.partial_decay(initial, final_state, env)
        except Exception as exc:
            partial_failures.append(
                {
                    "state": str(final_state),
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                }
            )
            continue
        if not _finite_float(gamma) or gamma == 0:
            continue
        rows.append(
            {
                "state": final_state,
                "label": str(final_state),
                "gamma_s^-1": float(gamma),
                "delta_GHz": float((initial.get_energy_Hz() - final_state.get_energy_Hz()) / 1e9),
                "F": float(final_state.f),
                "mF": float(final_state.m),
                "parity": int(final_state.parity),
            }
        )

    rows.sort(key=lambda row: row["gamma_s^-1"], reverse=True)
    total_gamma = float(sum(row["gamma_s^-1"] for row in rows))
    return {
        "basis": basis,
        "rows": rows,
        "partial_failures": partial_failures,
        "partial_seconds": time.perf_counter() - start,
        "total_gamma_s^-1": total_gamma,
        "tau_s": math.inf if total_gamma == 0 else 1.0 / total_gamma,
    }


def windowed_get_nearby_total_decay(atom, initial, env, **basis_kwargs):
    """Notebook-local replacement for total_decay using windowed get_nearby final states."""
    basis = build_windowed_get_nearby_basis(atom, initial, **basis_kwargs)
    return decay_from_windowed_basis(atom, initial, env, basis)


In [3]:
NU_MIN = 3.0
NU_MAX = initial_state.nub + 10.0
WINDOW_WIDTH = 5.0
WINDOW_OVERLAP = 0.10

print("Windowed get_nearby basis settings")
print(f"  nu range      = [{NU_MIN:.2f}, {NU_MAX:.8f}]")
print(f"  window width  = {WINDOW_WIDTH:.2f}")
print(f"  overlap       = {WINDOW_OVERLAP:.2f}")
print("  dl, df, dm    = 1, 1, 1")
print("  dipole filter = True")


Windowed get_nearby basis settings
  nu range      = [3.00, 74.56106363]
  window width  = 5.00
  overlap       = 0.10
  dl, df, dm    = 1, 1, 1
  dipole filter = True


In [4]:
basis = build_windowed_get_nearby_basis(
    yb,
    initial_state,
    nu_min=NU_MIN,
    nu_max=NU_MAX,
    window_width=WINDOW_WIDTH,
    window_overlap=WINDOW_OVERLAP,
    dl=1,
    df=1,
    dm=1,
    dipole_allowed=True,
)

print("Windowed basis")
print(f"  windows queried       = {len(basis['windows'])}")
print(f"  unique final states   = {len(basis['states'])}")
print(f"  get_nearby time       = {basis['seconds']:.2f} s")
print(f"  get_nearby errors     = {len(basis['errors'])}")
print(f"  skipped summary       = {basis['skipped']}")
print()

results = {}
for temperature_k in (0.0, 300.0):
    env = rydcalc.environment(T_K=temperature_k)
    result = decay_from_windowed_basis(yb, initial_state, env, basis)
    results[temperature_k] = result

    print(f"T = {temperature_k:g} K")
    print(f"  partial_decay time    = {result['partial_seconds']:.2f} s")
    print(f"  partial failures      = {len(result['partial_failures'])}")
    print(f"  nonzero channels      = {len(result['rows'])}")
    print(f"  Gamma_total           = {result['total_gamma_s^-1']:.6e} s^-1")
    print(f"  tau                   = {1e6 * result['tau_s']:.3f} us")
    print()


Windowed basis
  windows queried       = 15
  unique final states   = 1680
  get_nearby time       = 36.46 s
  get_nearby errors     = 0
  skipped summary       = {'nonfinite_energy': 0, 'not_dipole_allowed': 986, 'duplicates': 167}

T = 0 K
  partial_decay time    = 48.04 s
  partial failures      = 0
  nonzero channels      = 1444
  Gamma_total           = 1.554823e+03 s^-1
  tau                   = 643.160 us

T = 300 K
  partial_decay time    = 0.63 s
  partial failures      = 0
  nonzero channels      = 1680
  Gamma_total           = 5.596472e+03 s^-1
  tau                   = 178.684 us



/home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization/rydcalc/rydcalc/MQDTclass.py:90: RuntimeWarning: invalid value encountered in scalar power
  return ((Ia - Ib) / self.RydConst_invcm + 1 / nub ** 2) ** (-1 / 2)


## 最大贡献通道

下面列出 300 K 下最大的若干个 partial decay channel。这个结果来自 `get_nearby` 动态生成的 MQDT 终态，不是手写的终态列表。


In [5]:
for temperature_k in (0.0, 300.0):
    rows = results[temperature_k]["rows"]
    print(f"Top channels at T = {temperature_k:g} K")
    for index, row in enumerate(rows[:20], start=1):
        print(
            f"  {index:2d}. gamma={row['gamma_s^-1']:.6e} s^-1, "
            f"delta={row['delta_GHz']:.3f} GHz, {row['label']}"
        )
    print()


Top channels at T = 0 K
   1. gamma=1.958988e+02 s^-1, delta=210042.170 GHz, |171Yb:3.95,L=1,F=2.5,-2.5>
   2. gamma=1.813806e+02 s^-1, delta=416305.051 GHz, |171Yb:2.81,L=1,F=1.5,-1.5>
   3. gamma=1.209204e+02 s^-1, delta=416305.051 GHz, |171Yb:2.81,L=1,F=1.5,-0.5>
   4. gamma=7.835954e+01 s^-1, delta=210042.170 GHz, |171Yb:3.95,L=1,F=2.5,-1.5>
   5. gamma=6.306797e+01 s^-1, delta=356236.091 GHz, |171Yb:3.04,L=1,F=0.5,-0.5>
   6. gamma=5.757277e+01 s^-1, delta=211597.669 GHz, |171Yb:3.94,L=1,F=1.5,-1.5>
   7. gamma=5.345073e+01 s^-1, delta=127089.304 GHz, |171Yb:5.07,L=1,F=2.5,-2.5>
   8. gamma=3.838184e+01 s^-1, delta=211597.669 GHz, |171Yb:3.94,L=1,F=1.5,-0.5>
   9. gamma=3.551426e+01 s^-1, delta=201455.043 GHz, |171Yb:4.03,L=1,F=0.5,-0.5>
  10. gamma=2.931916e+01 s^-1, delta=87910.126 GHz, |171Yb:6.09,L=1,F=2.5,-2.5>
  11. gamma=2.647914e+01 s^-1, delta=90583.103 GHz, |171Yb:6.00,L=1,F=1.5,-1.5>
  12. gamma=2.232208e+01 s^-1, delta=129271.291 GHz, |171Yb:5.03,L=1,F=0.5,-0.5>
  13. 

## 和原生 `total_decay` 的关系

这个 notebook-local 函数修复的是 `total_decay` 的候选生成问题：

- 保留：`get_nearby` 动态生成终态；
- 保留：逐个终态调用 `partial_decay` 并求和；
- 改变：把一次性的巨大 MQDT `nu` 区间改成小窗口；
- 增加：窗口重叠去重，避免同一终态重复计数。

需要注意：这仍然只使用 `get_nearby` 返回的 MQDT 终态。它没有额外混入 `Yb171_NIST.txt` 中的低 `n P` 实验能级。因此它是对 `total_decay` 自动终态生成逻辑的局部修复测试，而不是最终的全物理 lifetime 推荐值。要逼近实验寿命，还需要把低 `n` NIST 终态和其他缺失 loss/BBR 通道作为额外候选或 residual sink 加入。
